## <a href="https://cursos.alura.com.br/course/langchain-desenvolva-agentes-inteligencia-artificial/task/161394?b2cUser=true"><b>Langchain Agentes - Criação do Agente com a Ferramenta</b></a><br/>

<b>Objetivo:</b> Criação do agente que chamará e executará a ferramenta.<br/>
<ul><li>Maneira pela qual a LLM toma ciência e executa a ferramenta.</li></ul>

<b>PASSOS:</b><br/>
<ul>
    <b><li>CRIAÇÃO DAS FERRAMENTAS</li></b><br/>
    <ul>
        <ol>
            <li>Criação da Ferramenta DadosDeEstudante (Refinando a anterior)</li>
            <li>Instanciando a Ferramenta que a LLM precisa usar</li>
        </ol>
    </ul><br/>
    <b><li>CRIAÇÃO DO AGENTE</li></b><br/>   
    <ul>
        <ol>
            <li>Informando para a LLM as ferramentas que eu tenho (Usa a ferramenta que foi instanciada)</li>
            <li>Executando o Agente com a ferramenta</li>
        </ol>
    </ul>
</ul>

In [1]:
#%pip install -r requirements.txt

In [2]:
from dotenv import load_dotenv
from os import getenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
#from langchain.globals import set_debug

#set_debug(True)

load_dotenv()

llm = ChatOpenAI(
                    model="gpt-5-mini",
                    api_key=getenv("API_KEY")            
                )

class ExtratordeEstudante(BaseModel):
    estudante: str = Field(description="Nome do estudante informado, sempre em letras minúsculas. Exemplo: joão, carla, joana")

Os dados dos estudantes estão em um arquivo CSV

In [3]:
from pandas import read_csv

def busca_dados_de_estudante(estudante) -> str:
    dfestudantes = read_csv("documentos/estudantes.csv")
    
    dados_estudante = dfestudantes.loc[dfestudantes['USUARIO'] == estudante]
    
    if dados_estudante.empty:
        return f"Desculpe, não encontrei dados para o estudante '{estudante}'. Por favor, verifique o nome e tente novamente."
    
    #print(estudante)
    
    return dados_estudante.to_string(index=False)

### <b>CRIAÇÃO DE FERRAMENTAS</b>
Que ferramentas eu tenho disponíveis para se obter os dados da Ana ?

<b>1) Criação da Ferramenta DadosDeEstudante</b>
<ul>
    <li> A classe deve estender de BaseTool</li>
    <li> Essa ferramenta deve ter nome, descrição e o método run estendido de BaseTool</li>
</ul>

In [4]:
from langchain.tools import BaseTool
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

class DadosDeEstudante(BaseTool):
    
    # TODA FERRAMENTA PRECISA TER ESSES ATRIBUTOS
    name: str = "dados_de_estudante" # Nome da ferramenta
    description : str = """ 
                            Essa ferramenta extrai o histórico e preferências de um estudante, de acordo com o seu histórico.
                        """ # Descrição da ferramenta 
    
    # CONTRATO DA FERRAMENTA - O QUE ELA FAZ
    def _run(self, input: str) -> str:        
        
        parseador = JsonOutputParser(pydantic_object=ExtratordeEstudante)    
        
        template = PromptTemplate(
                                    template = """ 
                                                    Você deve analisar a {input} e extrair o nome de estudante informado.
                                            
                                                    FORMATO DE SAIDA:
                                                    {formato_saida} 
                                                """,
                                    input_variables = ["input"],
                                    partial_variables = {"formato_saida": parseador.get_format_instructions()}
                                 )
        
        cadeia = template | llm | parseador
        
        resposta = cadeia.invoke({"input": input})
        
        estudante = resposta['estudante']
        
        return busca_dados_de_estudante(estudante)


<b>2) Instanciando a Ferramenta que a LLM precisa usar</b>
<ul>
    <li> Para o objeto da classe Tool, deverão ser informados o descrição e a função, o método run da ferramenta que foi criada.</li>
</ul>

In [5]:
from langchain.agents import Tool

dados_de_estudante = DadosDeEstudante() # INSTANCIANDO O OBJETO DA MINHA FERRAMENTA

# MATRIZ DE FERRAMENTAS (CONJUNTO DE FERRAMENTAS)
tools = [
            # Instanciando ferramentas
            Tool(
                    name=dados_de_estudante.name,
                    func=dados_de_estudante.run,
                    description=dados_de_estudante.description                
            )
]

### <b>CRIAÇÃO DO AGENTE</b>

<b>3) Informando para a LLM as ferramentas que eu tenho</b> 
<ul><li>Para isso, é necessário criar um agente com as ferramentas</li></ul>

In [6]:
from langchain.agents import create_openai_tools_agent
from langchain import hub
import warnings

warnings.filterwarnings("ignore")


# PROMPT DE INICIALIZAÇÃO PARA INFORMAR PARA A LLM SOBRE A FERRAMENTA.
prompt=(hub.pull(owner_repo_commit="hwchase17/openai-functions-agent"))

# CRIANDO UM AGENTE COM AS FERRAMENTAS
agente = create_openai_tools_agent(
                                    llm=llm, # INFORMA A LLM QUE VAI SER USADA PELO AGENTE
                                    tools=tools, # PASSANDO PARA A LLM A FERRAMENTA QUE ELA PODE USAR. INSTÂNCIA DA FERRAMENTA
                                    prompt=prompt  
                                                  # JÁ EXISTEM PROMPTS PRONTOS NO REPOSITÓRIO DO LANGSMITH, DE ACORDO COM O TIPO DE FERRAMENTA. 
                                                        # Para agente de função (https://smith.langchain.com/hub/hwchase17/openai-functions-agent)
                                                                 
                                  )

print(prompt)


input_variables=['agent_scratchpad', 'input'] optional_variables=['chat_history'] input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')]

<b>4) Executando o agente com a ferramenta</b>

In [7]:
from langchain.agents import AgentExecutor

executor = AgentExecutor(
                            agent=agente, # O AGENTE QUE VAI SER USADO
                            tools=tools, # FERRAMENTAS QUE O AGENTE PODE USAR
                            verbose=True
                        )

pergunta = "Quais são os dados do Ana?"

resposta = executor.invoke({"input": pergunta})

print(resposta)



> Entering new AgentExecutor chain...

Invoking: `dados_de_estudante` with `Ana`


NOME USUARIO  ANO_FORMATURA  SCORE_MATEMATICA  SCORE_PORTUGUES_  SCORE_BIOLOGIA  SCORE_FISICA  SCORE_COMPUTACAO  SCORE_FILOSOFIA  SCORE_PROJETOS  SCORE_ATIVIDADES_SOCIAIS  SCORE_PUBLICACOES LISTA_AREAS_PREFERIDAS  PROEFICIENCIA_INGLES  PROEFICIENCIA_ESPANHOL LISTA_PAISES_PREFERIDOS                LISTA_UNIVERSIDADES_PREFERIDAS
 Ana     ana           2029                 4                 6               8             8                 3                5               5                         2                  0            ['Humanas']                   5.5                     5.5  ['Brasil', 'Alemanha'] ['UNICAMP', 'Technical University of Berlin']Aqui estão os dados da Ana:

- Nome: Ana  
- Usuário: ana  
- Ano de formatura: 2029

- Scores:
  - Matemática: 4
  - Português: 6
  - Biologia: 8
  - Física: 8
  - Computação: 3
  - Filosofia: 5
  - Projetos: 5
  - Atividades sociais: 2
  - Publicações: 0



A saída "ana" (Em azul), já comprova que a LLM usou a nossa ferramenta